# RuSearchRank Phase 1A — full MIRACL Russian BM25

This notebook is a thin Linux/Colab wrapper around `rusearchrank.cli`. It downloads the official 6.42 GiB compressed Lucene index, runs train top-100 and the official dev top-1000, evaluates the untouched dev run, then creates a separate stable dev top-100 run and the project candidate cache while streaming only the selected passages. It creates `artifacts/rusearchrank_phase1_results.zip`. Python dependencies and the extracted index need additional temporary space; the runner requires at least 30 GiB free.

The Lucene index, corpus/Hugging Face/Pyserini caches, Python environment, `.git`, and temporary work files are not included in the ZIP and must not be added to Git. The ZIP contains only candidate Parquet files, train top-100, raw dev top-1000, derived dev top-100, and the three Phase 1 audit JSON files. Run all 14 cells in order; the heavy operations are deliberately separated.

In [ ]:
# Cell 2 — fail-fast Linux, hardware, and disk gate (no downloads).
import os, platform, shutil, sys
from pathlib import Path

MIN_FREE_GIB = 30
disk = shutil.disk_usage('/content' if Path('/content').is_dir() else '.')
mem_kib = next((int(line.split()[1]) for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:')), 0) if Path('/proc/meminfo').is_file() else 0
environment = {
    'os': platform.platform(),
    'python': platform.python_version(),
    'cpu': platform.processor() or platform.machine(),
    'cpu_count': os.cpu_count(),
    'ram_gib': round(mem_kib / 1024**2, 2),
    'disk_free_gib': round(disk.free / 1024**3, 2),
}
print(environment)
if platform.system() != 'Linux':
    raise RuntimeError('This runner must execute on Linux (Google Colab is supported).')
if disk.free < MIN_FREE_GIB * 1024**3:
    raise RuntimeError(f'Insufficient free disk: {environment["disk_free_gib"]} GiB; at least {MIN_FREE_GIB} GiB is required before downloading the index.')

In [ ]:
# Cell 3 — one diagnostic command helper; clone/fast-forward the exact branch.
import json, os, shlex, subprocess, sys, threading
from pathlib import Path

def run_checked(command, *, cwd=None, env=None, stream=False, log_path=None):
    command = [str(part) for part in command]
    effective_cwd = Path(cwd or Path.cwd()).resolve()
    effective_env = os.environ.copy() if env is None else env.copy()
    print('+', shlex.join(command), f'(cwd={effective_cwd})', flush=True)
    try:
        if stream:
            process = subprocess.Popen(command, cwd=effective_cwd, env=effective_env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)
            stdout_lines, stderr_lines = [], []
            def pump(pipe, destination, collected):
                for line in iter(pipe.readline, ''):
                    collected.append(line)
                    print(line, end='', file=destination, flush=True)
                pipe.close()
            workers = [threading.Thread(target=pump, args=(process.stdout, sys.stdout, stdout_lines), daemon=True), threading.Thread(target=pump, args=(process.stderr, sys.stderr, stderr_lines), daemon=True)]
            for worker in workers: worker.start()
            returncode = process.wait()
            for worker in workers: worker.join()
            stdout, stderr = ''.join(stdout_lines), ''.join(stderr_lines)
        else:
            result = subprocess.run(command, cwd=effective_cwd, env=effective_env, text=True, capture_output=True, check=False)
            returncode, stdout, stderr = result.returncode, result.stdout, result.stderr
            if stdout: print(stdout, end='' if stdout.endswith('\n') else '\n')
            if stderr: print(stderr, end='' if stderr.endswith('\n') else '\n', file=sys.stderr)
    except OSError as exc:
        returncode, stdout, stderr = None, '', str(exc)
        print('stdout:', stdout, file=sys.stderr)
        print('stderr:', stderr, file=sys.stderr)
    record = {'command': command, 'cwd': str(effective_cwd), 'returncode': returncode, 'stdout': stdout, 'stderr': stderr}
    if log_path:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text(json.dumps(record, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
        print('complete log:', log_path)
    if returncode != 0:
        raise RuntimeError(f'command failed with return code {returncode}: {shlex.join(command)}' + (f'; complete log: {log_path}' if log_path else ''))
    return record

REPO_URL = 'https://github.com/kopanevk/ru-search-rank.git'
BRANCH = 'phase-0'
REPO_DIR = Path('/content/ru-search-rank')
ALLOW_OVERWRITE_RUNS_AND_CACHE = False  # Set True only for an intentional rerun.
if (REPO_DIR / '.git').is_dir():
    run_checked(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR)
    run_checked(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run_checked(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR)
else:
    run_checked(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], cwd='/content')
os.chdir(REPO_DIR)
run_checked(['git', 'log', '-1', '--oneline', '--decorate'], cwd=REPO_DIR)

In [ ]:
# Cell 4 — install Java 21 and pinned official NIST trec_eval v9.0.8.
import glob, os, re, subprocess
from pathlib import Path

run_checked(['apt-get', 'update', '-qq'])
run_checked(['apt-get', 'install', '-y', '-qq', 'openjdk-21-jdk-headless', 'build-essential'])
java_candidates = sorted(glob.glob('/usr/lib/jvm/java-21*/bin/java'))
if not java_candidates:
    raise RuntimeError('OpenJDK 21 was installed but its java executable was not found.')
java_bin = Path(java_candidates[0]).resolve()
JAVA_HOME = java_bin.parent.parent
os.environ['JAVA_HOME'] = str(JAVA_HOME)
os.environ['PATH'] = f'{java_bin.parent}:' + os.environ['PATH']
java_probe = run_checked([str(java_bin), '-version'], env=os.environ)
java_version = java_probe['stderr'] or java_probe['stdout']
print(java_version)
if not re.search(r'(?:version\s+"|openjdk\s+)21(?:[."]|$)', java_version.lower()):
    raise RuntimeError(f'Active Java is not 21; JAVA_HOME={JAVA_HOME}')

TREC_EVAL_TAG = 'v9.0.8'
TREC_EVAL_DIR = Path('/content/trec_eval-v9.0.8')
if not (TREC_EVAL_DIR / '.git').is_dir():
    run_checked(['git', 'clone', '--depth', '1', '--branch', TREC_EVAL_TAG, 'https://github.com/usnistgov/trec_eval.git', str(TREC_EVAL_DIR)], cwd='/content')
else:
    run_checked(['git', 'fetch', '--tags', 'origin'], cwd=TREC_EVAL_DIR)
    run_checked(['git', 'checkout', '--detach', TREC_EVAL_TAG], cwd=TREC_EVAL_DIR)
trec_head = run_checked(['git', 'rev-parse', 'HEAD'], cwd=TREC_EVAL_DIR)['stdout'].strip()
trec_tag_commit = run_checked(['git', 'rev-list', '-n', '1', TREC_EVAL_TAG], cwd=TREC_EVAL_DIR)['stdout'].strip()
if trec_head != trec_tag_commit:
    raise RuntimeError(f'trec_eval checkout is not exactly {TREC_EVAL_TAG}: {trec_head}')
run_checked(['make', '-j', str(os.cpu_count() or 2)], cwd=TREC_EVAL_DIR)
run_checked(['install', '-m', '0755', str(TREC_EVAL_DIR / 'trec_eval'), '/usr/local/bin/trec_eval'])
trec_eval_probe = run_checked(['/usr/local/bin/trec_eval', '-h'])
print('trec_eval expected version', TREC_EVAL_TAG, trec_eval_probe['stdout'] or trec_eval_probe['stderr'])

In [ ]:
# Cell 5 — create an isolated Python 3.12 environment without replacing system Python.
import shutil, subprocess, sys
from pathlib import Path

VENV_DIR = Path('/content/rusearchrank-py312')
if sys.version_info[:2] == (3, 12):
    run_checked(['apt-get', 'install', '-y', '-qq', 'python3.12-venv'])
    python312 = Path(sys.executable)
    if not (VENV_DIR / 'bin/python').is_file():
        run_checked([str(python312), '-m', 'venv', str(VENV_DIR)])
else:
    UV_VERSION = '0.8.13'
    uv_prefix = Path('/content/uv-bootstrap')
    uv = uv_prefix / 'bin/uv'
    if not uv.is_file():
        run_checked([sys.executable, '-m', 'pip', 'install', '--prefix', str(uv_prefix), f'uv=={UV_VERSION}'])
    run_checked([str(uv), 'python', 'install', '3.12'])
    if not (VENV_DIR / 'bin/python').is_file():
        run_checked([str(uv), 'venv', '--python', '3.12', str(VENV_DIR)])
RUN_PYTHON = VENV_DIR / 'bin/python'
python_probe = run_checked([str(RUN_PYTHON), '--version'])
actual_python = (python_probe['stdout'] or python_probe['stderr']).strip()
print(actual_python, RUN_PYTHON)
if not actual_python.startswith('Python 3.12.'):
    raise RuntimeError(f'Isolated interpreter is not Python 3.12: {actual_python}')

In [ ]:
# Cell 6 — install the project and pinned retrieval dependency, then test under Python 3.12.
import os, subprocess

run_checked([str(RUN_PYTHON), '-m', 'pip', 'install', '--upgrade', 'pip'], cwd=REPO_DIR)
run_checked([str(RUN_PYTHON), '-m', 'pip', 'install', '-e', f'{REPO_DIR}[retrieval]'], cwd=REPO_DIR)
version_probe = "import importlib.metadata as m,sys; print(sys.version); print('pyserini',m.version('pyserini'))"
run_checked([str(RUN_PYTHON), '-c', version_probe], cwd=REPO_DIR, env=os.environ)
# Exact isolated-environment equivalent of: python -m pytest -q
run_checked([str(RUN_PYTHON), '-m', 'pytest', '-q'], cwd=REPO_DIR, env=os.environ)

In [ ]:
# Cell 7 — download official topics/qrels, then run full retrieval preflight/index smoke.
import os, re, subprocess

CONFIG = 'configs/retrieval.yaml'
def cli(*arguments, stream=False):
    command = [str(RUN_PYTHON), '-m', 'rusearchrank.cli', *arguments]
    log_name = re.sub(r'[^A-Za-z0-9_.-]+', '_', '__'.join(map(str, arguments))) + '.json'
    log_path = REPO_DIR / 'artifacts/work/phase1/notebook_logs' / log_name
    return run_checked(command, cwd=REPO_DIR, env=os.environ, stream=stream, log_path=log_path)

cli('prepare-annotations', '--config', CONFIG)
cli('preflight', '--config', CONFIG, '--stage', 'retrieval', '--check-index', stream=True)

In [ ]:
# Cell 8 — train BM25 top-100 from the validated local official TSV (4,683 queries).
overwrite = ['--overwrite'] if ALLOW_OVERWRITE_RUNS_AND_CACHE else []
cli('run-bm25', '--config', CONFIG, '--split', 'train', *overwrite, stream=True)

In [ ]:
# Cell 9 — official dev BM25 with --hits 1000; preserve the raw run for reproduction.
overwrite = ['--overwrite'] if ALLOW_OVERWRITE_RUNS_AND_CACHE else []
cli('run-bm25', '--config', CONFIG, '--split', 'dev', *overwrite, stream=True)

In [ ]:
# Cell 10 — evaluate the untouched dev top-1000 with official trec_eval commands.
# run_checked stops here with command/stdout/stderr/return code unless the gate passes.
cli('evaluate-bm25', '--config', CONFIG, '--overwrite')

In [ ]:
# Cell 11 — preflight, idempotent stable dev top-100, three-state cache, passages, qrels audit.
overwrite = ['--overwrite'] if ALLOW_OVERWRITE_RUNS_AND_CACHE else []
cli('preflight', '--config', CONFIG, '--stage', 'candidate-cache')
cli('build-candidate-cache', '--config', CONFIG, *overwrite, stream=True)
cli('audit-qrels', '--config', CONFIG, '--overwrite')

In [ ]:
# Cell 12 — candidate, query, passage, three-state, top-K, and stable-order validation.
cli('validate-candidates', 'artifacts/candidates/train_top100.parquet', '--config', CONFIG)
cli('validate-candidates', 'artifacts/candidates/dev_top100.parquet', '--config', CONFIG)
cli('preflight', '--config', CONFIG, '--stage', 'package')

In [ ]:
# Cell 13 — package the explicit portable allowlist; caches/index/environment are excluded.
cli('package-phase1', '--config', CONFIG, '--overwrite', stream=True)

In [ ]:
# Cell 14 — show ZIP size, SHA-256, exact contents, then download (optional Drive copy).
import hashlib, shutil, zipfile
from pathlib import Path

archive_path = REPO_DIR / 'artifacts/rusearchrank_phase1_results.zip'
digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
with zipfile.ZipFile(archive_path) as archive:
    contents = archive.namelist()
print({'path': str(archive_path), 'size_bytes': archive_path.stat().st_size, 'sha256': digest, 'contents': contents})

DRIVE_DESTINATION = ''  # Optional, e.g. '/content/drive/MyDrive/rusearchrank_phase1_results.zip'.
if DRIVE_DESTINATION:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copy2(archive_path, DRIVE_DESTINATION)
    print('Copied to', DRIVE_DESTINATION)
try:
    from google.colab import files
except ImportError:
    print('Not running in Colab; retrieve the validated ZIP from', archive_path)
else:
    files.download(str(archive_path))